In [1]:
import torch
import torch.nn as nn

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim

        self.wf = nn.Parameter(torch.randn(hidden_dim, concat_dim))
        self.wi = nn.Parameter(torch.randn(hidden_dim, concat_dim))
        self.wc = nn.Parameter(torch.randn(hidden_dim, concat_dim)) 
        self.wo = nn.Parameter(torch.randn(hidden_dim, concat_dim))

        self.bf = nn.Parameter(torch.randn(hidden_dim, 1))
        self.bi = nn.Parameter(torch.randn(hidden_dim, 1))
        self.bc = nn.Parameter(torch.randn(hidden_dim, 1))
        self.bo = nn.Parameter(torch.randn(hidden_dim, 1))

    def forward(self, x):
        seq_len = x.size(0)
        h = torch.zeros(self.hidden_dim, 1)
        c = torch.zeros(self.hidden_dim, 1)

        encoder_outputs = []
        for t in range(seq_len):
            x_t = x[t].view(-1, 1)  # Typo fixed
            combined = torch.cat((h, x_t), dim=0)

            ft = torch.sigmoid(self.wf @ combined + self.bf)
            it = torch.sigmoid(self.wi @ combined + self.bi)
            c_tilde = torch.tanh(self.wc @ combined + self.bc)
            c = ft * c + it * c_tilde

            ot = torch.sigmoid(self.wo @ combined + self.bo)
            h = ot * torch.tanh(c)
            encoder_outputs.append(h)
            
        return torch.stack(encoder_outputs), h, c

In [3]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.U = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.v = nn.Parameter(torch.randn(1, hidden_dim))

        # Gradient accumulators
        self.dW = torch.zeros_like(self.W)
        self.dU = torch.zeros_like(self.U)
        self.dv = torch.zeros_like(self.v)

    def forward(self, dec_hidden, encoder_outputs):
        seq_len = encoder_outputs.size(0)
        scores = torch.zeros(seq_len, 1)
        
        z_list = []
        energy_list = []

        for t in range(seq_len):
            enc_h = encoder_outputs[t].view(-1, 1)
            z = self.W @ dec_hidden + self.U @ enc_h
            
            z_list.append(z)
            energy = self.v @ torch.tanh(z)
            scores[t] = energy[0]
            energy_list.append(torch.tanh(z))

        alphas = torch.softmax(scores, dim=0)
        
        context_vector = torch.zeros_like(dec_hidden)
        for t in range(seq_len):
            context_vector += alphas[t] * encoder_outputs[t].view(-1, 1)
            
        # Cache variables needed for backprop
        cache = (dec_hidden, encoder_outputs, z_list, energy_list, alphas)
        return context_vector, alphas, cache

    def backward(self, dcontext, cache):
        dec_hidden, encoder_outputs, z_list, energy_list, alphas = cache
        seq_len = encoder_outputs.size(0)
        
        d_alphas = torch.zeros(seq_len, 1)
        for t in range(seq_len):
            enc_h = encoder_outputs[t].view(-1, 1)
            d_alphas[t] = (dcontext.t() @ enc_h).squeeze()

        # Complex Softmax Derivative
        sum_alpha_dalpha = torch.sum(alphas * d_alphas)
        d_scores = alphas * (d_alphas - sum_alpha_dalpha)

        d_dec_hidden = torch.zeros_like(dec_hidden)
        d_encoder_outputs = torch.zeros_like(encoder_outputs)

        for t in range(seq_len):
            enc_h = encoder_outputs[t].view(-1, 1)
            z = z_list[t]
            de = d_scores[t]
            
            # Derivative of tanh(z) is (1 - tanh^2(z))
            dz = de * self.v.t() * (1 - energy_list[t]**2)
            
            # Accumulate gradients for learnable weights
            self.dv += de * energy_list[t].t()
            self.dW += dz @ dec_hidden.t()
            self.dU += dz @ enc_h.t()
            
            # Pass gradients back to inputs
            d_dec_hidden += self.W.t() @ dz
            d_enc_h = alphas[t] * dcontext + self.U.t() @ dz
            d_encoder_outputs[t] = d_enc_h.view_as(encoder_outputs[t])

        return d_dec_hidden, d_encoder_outputs

In [4]:
class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention = Attention(hidden_dim)

        concat_dim = input_dim + hidden_dim + hidden_dim

        self.wf = nn.Parameter(torch.randn(hidden_dim, concat_dim))
        self.wi = nn.Parameter(torch.randn(hidden_dim, concat_dim))
        self.wc = nn.Parameter(torch.randn(hidden_dim, concat_dim))
        self.wo = nn.Parameter(torch.randn(hidden_dim, concat_dim))
        self.wy = nn.Parameter(torch.randn(1, hidden_dim)) 

        self.bf = nn.Parameter(torch.randn(hidden_dim, 1))
        self.bi = nn.Parameter(torch.randn(hidden_dim, 1))
        self.bc = nn.Parameter(torch.randn(hidden_dim, 1))
        self.bo = nn.Parameter(torch.randn(hidden_dim, 1))
        self.by = nn.Parameter(torch.randn(1, 1))

    def forward(self, x, h_prev, c_prev, encoder_outputs):
        seq_len = x.size(0)
        h, c = h_prev, c_prev

        predictions = []
        self.h_list, self.c_list, self.x_list = [], [], []
        self.f_list, self.i_list, self.c_tilde_list, self.o_list = [], [], [], []
        self.y_list, self.context_list, self.attn_caches = [], [], []

        for t in range(seq_len):
            x_t = x[t].view(-1, 1)
            
            context, alphas, cache = self.attention(h, encoder_outputs)
            self.attn_caches.append(cache)
            self.context_list.append(context)

            h_prev_t = h # Capture pre-update state for backprop
            combined = torch.cat((h_prev_t, context, x_t), dim=0)

            ft = torch.sigmoid(self.wf @ combined + self.bf)
            it = torch.sigmoid(self.wi @ combined + self.bi)
            c_tilde = torch.tanh(self.wc @ combined + self.bc)
            c = ft * c + it * c_tilde

            ot = torch.sigmoid(self.wo @ combined + self.bo)
            h = ot * torch.tanh(c)
            y_t = torch.sigmoid(self.wy @ h + self.by)
            
            predictions.append(y_t)
            
            # Caching
            self.h_list.append(h)
            self.c_list.append(c)
            self.x_list.append(x_t)
            self.f_list.append(ft)
            self.i_list.append(it)
            self.c_tilde_list.append(c_tilde)
            self.o_list.append(ot)
            self.y_list.append(y_t)

        return predictions, h, c

    def backward(self, x, y):
        h_future = torch.zeros(self.hidden_dim, 1)
        c_future = torch.zeros(self.hidden_dim, 1)

        self.dw_f, self.dw_i, self.dw_c, self.dw_o = (torch.zeros_like(self.wf), torch.zeros_like(self.wi), 
                                                      torch.zeros_like(self.wc), torch.zeros_like(self.wo))
        self.dw_y = torch.zeros_like(self.wy)

        self.db_f, self.db_i, self.db_c, self.db_o = (torch.zeros_like(self.bf), torch.zeros_like(self.bi), 
                                                      torch.zeros_like(self.bc), torch.zeros_like(self.bo))
        self.db_y = torch.zeros_like(self.by)

        # Reset attention accumulated gradients
        self.attention.dW.zero_()
        self.attention.dU.zero_()
        self.attention.dv.zero_()

        for t in reversed(range(x.size(0))):
            x_t = self.x_list[t]
            h_t = self.h_list[t]
            c_t = self.c_list[t]
            y_t = self.y_list[t]
            o_t = self.o_list[t]
            c_tilde_t = self.c_tilde_list[t]
            context_t = self.context_list[t]

            # Fetch correct previous states
            h_prev = self.h_list[t-1] if t > 0 else torch.zeros(self.hidden_dim, 1)
            c_prev = self.c_list[t-1] if t > 0 else torch.zeros(self.hidden_dim, 1)

            dy = y_t - y[t].view(-1, 1)
            dh = self.wy.t() @ dy + h_future

            self.dw_y += dy @ h_t.t()
            self.db_y += dy

            do = dh * torch.tanh(c_t)
            dc = dh * o_t * (1 - torch.tanh(c_t) ** 2) + c_future
            di = dc * c_tilde_t
            df = dc * c_prev 

            dzf = df * self.f_list[t] * (1 - self.f_list[t])
            dzi = di * self.i_list[t] * (1 - self.i_list[t])
            dzc_tilde = dc * self.i_list[t] * (1 - c_tilde_t ** 2)
            dzo = do * o_t * (1 - o_t)

            combined = torch.cat((h_prev, context_t, x_t), dim=0) 

            self.dw_f += dzf @ combined.t()
            self.dw_i += dzi @ combined.t()
            self.dw_c += dzc_tilde @ combined.t()
            self.dw_o += dzo @ combined.t()

            self.db_f += dzf
            self.db_i += dzi
            self.db_c += dzc_tilde
            self.db_o += dzo

            dz_concat = (self.wf.t() @ dzf + 
                         self.wi.t() @ dzi + 
                         self.wc.t() @ dzc_tilde + 
                         self.wo.t() @ dzo)

            # Split gradients to route them correctly
            dh_from_cell = dz_concat[:self.hidden_dim, :]
            dcontext = dz_concat[self.hidden_dim:2*self.hidden_dim, :]

            # Push context gradient backward into the Attention module
            d_dec_hidden_attn, d_enc_out_attn = self.attention.backward(dcontext, self.attn_caches[t])

            # The total gradient for h_prev comes from the cell operations AND the attention query
            h_future = dh_from_cell + d_dec_hidden_attn
            c_future = dc * self.f_list[t]

In [5]:
# 1. Define hyperparameters
seq_len = 5
input_dim = 3
hidden_dim = 4

# Set random seed for reproducibility during testing
torch.manual_seed(42)
# 2. Initialize the models
encoder = Encoder(input_dim, hidden_dim)
decoder = Decoder(input_dim, hidden_dim)
# 3. Create dummy data
# Encoder input: (seq_len, input_dim)
x_enc = torch.randn(seq_len, input_dim) 

# Decoder input (e.g., shifted target sequence): (seq_len, input_dim)
x_dec = torch.randn(seq_len, input_dim) 

# Target output: (seq_len, 1) because wy produces a (1, 1) output per step
y_target = torch.rand(seq_len, 1)       
print("--- Starting Forward Pass ---")

# 4. Encoder Forward Pass
encoder_outputs, h_enc, c_enc = encoder(x_enc)
print(f"Encoder outputs shape: {encoder_outputs.shape}")

# 5. Decoder Forward Pass (using Encoder's final states as initial Decoder states)
predictions, h_dec, c_dec = decoder(x_dec, h_enc, c_enc, encoder_outputs)
print(f"Generated {len(predictions)} predictions.")
print("\n--- Starting Backward Pass ---")

# 6. Decoder Backward Pass (this triggers Attention.backward() internally)
decoder.backward(x_dec, y_target)
print("Backward pass completed successfully.")
print("\n--- Verifying Gradients ---")

# 7. Verify Gradients in Decoder
print("Decoder Output Weights (dw_y) gradient sum:", decoder.dw_y.sum().item())
print("Decoder Forget Gate (dw_f) gradient sum:", decoder.dw_f.sum().item())

# 8. Verify Gradients in Attention Mechanism
print("Attention Query Weights (dW) gradient sum:", decoder.attention.dW.sum().item())
print("Attention Key Weights (dU) gradient sum:", decoder.attention.dU.sum().item())
print("Attention Energy Vector (dv) gradient sum:", decoder.attention.dv.sum().item())

# Basic assertions to ensure gradients are actually populated (not exactly zero)
assert decoder.dw_y.abs().sum() > 0, "Decoder output gradients are zero!"
assert decoder.attention.dW.abs().sum() > 0, "Attention W gradients did not update!"
assert decoder.attention.dU.abs().sum() > 0, "Attention U gradients did not update!"

print("\nAll gradient checks passed. The chain rule is fully connected!")

--- Starting Forward Pass ---
Encoder outputs shape: torch.Size([5, 4, 1])
Generated 5 predictions.

--- Starting Backward Pass ---
Backward pass completed successfully.

--- Verifying Gradients ---
Decoder Output Weights (dw_y) gradient sum: -0.7623728513717651
Decoder Forget Gate (dw_f) gradient sum: -0.07948831468820572
Attention Query Weights (dW) gradient sum: 0.001139674917794764
Attention Key Weights (dU) gradient sum: 0.001173939323052764
Attention Energy Vector (dv) gradient sum: -0.0068961698561906815

All gradient checks passed. The chain rule is fully connected!
